In [47]:
import numpy as np
import pandas as pd
import seaborn as sns

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

## 전처리

In [48]:
train['Amount'] = np.log1p(train['Amount'])
test['Amount']  = np.log1p(test['Amount'])

train['hour'] = (train['Time']//3600)%24
test['hour']  = (test['Time']//3600)%24

train.drop('Time', axis=1, inplace=True)
test.drop('Time', axis=1, inplace=True)

thr = train['Amount'].quantile(0.95)

# 
train['high_amount'] = (train['Amount'] > thr).astype(int)
test['high_amount'] = (test['Amount'] > thr).astype(int)

## 데이터 분할

In [49]:
X = train.drop(['Class','id'], axis=1)
y = train['Class']

X_test = test.drop('id', axis=1) # 테스트 데이터

# 교차검증용 KFold
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [50]:
import numpy as np

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from xgboost import XGBClassifier

# 튜닝용 CV (빠르게)
skf_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# threshold 후보 (너무 촘촘하면 오래 걸림)
THRESHOLDS = np.arange(0.15, 0.61, 0.03)


def cv_f1_threshold_xgb(params, X, y, cv):
    scores = []

    for tr_idx, va_idx in cv.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = XGBClassifier(
            n_estimators=int(params['n_estimators']),
            max_depth=int(params['max_depth']),
            learning_rate=float(params['learning_rate']),
            subsample=float(params['subsample']),
            colsample_bytree=float(params['colsample_bytree']),
            min_child_weight=float(params['min_child_weight']),
            gamma=float(params['gamma']),
            reg_lambda=float(params['reg_lambda']),
            reg_alpha=float(params['reg_alpha']),
            scale_pos_weight=float(params['scale_pos_weight']),
            tree_method='hist',
            n_jobs=-1,
            random_state=42,
            eval_metric='logloss'
        )

        model.fit(X_tr, y_tr)

        proba = model.predict_proba(X_va)[:, 1]

        best = 0.0
        for t in THRESHOLDS:
            pred = (proba >= t).astype(int)
            f1 = f1_score(y_va, pred)
            if f1 > best:
                best = f1

        scores.append(best)

    return float(np.mean(scores)), float(np.std(scores))


# scale_pos_weight 기본값(탐색 범위의 중심으로 씀)
scale_base = (y == 0).sum() / (y == 1).sum()

space = {
    'n_estimators': hp.quniform('n_estimators', 400, 1400, 50),
    'max_depth': hp.quniform('max_depth', 3, 10, 1),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'min_child_weight': hp.uniform('min_child_weight', 1.0, 10.0),
    'gamma': hp.uniform('gamma', 0.0, 5.0),
    'reg_lambda': hp.uniform('reg_lambda', 0.0, 10.0),
    'reg_alpha': hp.uniform('reg_alpha', 0.0, 5.0),
    'scale_pos_weight': hp.uniform('scale_pos_weight', 0.5 * scale_base, 1.5 * scale_base)
}

trials = Trials()

def objective(params):
    mean_f1, std_f1 = cv_f1_threshold_xgb(params, X, y, skf_tune)
    return {
        'loss': -mean_f1,
        'status': STATUS_OK,
        'mean_f1': mean_f1,
        'std_f1': std_f1
    }

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=40,
    trials=trials,
    rstate=np.random.default_rng(42)
)

best_params = {
    'n_estimators': int(best['n_estimators']),
    'max_depth': int(best['max_depth']),
    'learning_rate': float(best['learning_rate']),
    'subsample': float(best['subsample']),
    'colsample_bytree': float(best['colsample_bytree']),
    'min_child_weight': float(best['min_child_weight']),
    'gamma': float(best['gamma']),
    'reg_lambda': float(best['reg_lambda']),
    'reg_alpha': float(best['reg_alpha']),
    'scale_pos_weight': float(best['scale_pos_weight'])
}

print(best_params)

# 최종 모델(이 파라미터로 전체 학습)
xgb_tuned = XGBClassifier(
    **best_params,
    tree_method='hist',
    n_jobs=-1,
    random_state=42,
    eval_metric='logloss'
)

100%|████████| 40/40 [02:41<00:00,  4.03s/trial, best loss: -0.8873029042520568]
{'n_estimators': 950, 'max_depth': 10, 'learning_rate': 0.19891309923529624, 'subsample': 0.7462846819125962, 'colsample_bytree': 0.6798578324959299, 'min_child_weight': 1.581081400830431, 'gamma': 0.22027574700959424, 'reg_lambda': 7.762580258579903, 'reg_alpha': 1.9685968536555132, 'scale_pos_weight': 348.3214380821764}


In [55]:
import joblib
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
# =========
# PART 1) 튜닝 파라미터로 모델 생성
# =========

params = best_params.copy()

params['scale_pos_weight'] = 1   # 🔥 필살 카드

xgb_no_weight = XGBClassifier(
    **params,
    tree_method='hist',
    n_jobs=-1,
    random_state=42,
    eval_metric='logloss'
)


# =========
# PART 2) (권장) OOF로 best threshold 찾기 + OOF F1/Recall 출력
# =========
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(y))

for tr_idx, va_idx in skf.split(X, y):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr = y.iloc[tr_idx]

    xgb_tuned.fit(X_tr, y_tr)
    oof[va_idx] = xgb_tuned.predict_proba(X_va)[:, 1]

thresholds = np.arange(0.05, 0.61, 0.01)

best_t = None
best_f1 = -1.0
best_recall = None

for t in thresholds:
    pred = (oof >= t).astype(int)
    f1 = f1_score(y, pred)
    if f1 > best_f1:
        best_f1 = f1
        best_t = float(t)
        best_recall = float(recall_score(y, pred))

print('best_t:', best_t, '/ best_oof_f1:', best_f1, '/ oof_recall:', best_recall)


# =========
# PART 3) 최종 학습(전체 train) + 모델 저장
# =========
xgb_tuned.fit(X, y)

joblib.dump(
    {
        'model': xgb_tuned,
        'best_params': best_params,
        'best_t': best_t,
        'features': list(X.columns)
    },
    'xgb_tuned_bundle.joblib'
)

print('saved: xgb_tuned_bundle.joblib')


# =========
# PART 4) 로드해서 그대로 예측/제출 (안전하게 feature 정렬 포함)
# =========
bundle = joblib.load('xgb_tuned_bundle.joblib')

model = bundle['model']
best_t = bundle['best_t']
features = bundle['features']

X_test_aligned = X_test[features]

test_proba = model.predict_proba(X_test_aligned)[:, 1]
test_pred = (test_proba >= best_t).astype(int)

submission = pd.DataFrame({'id': test['id'], 'Class': test_pred})
submission.to_csv('submission_xgb_tuned.csv', index=False)

print('fraud rate:', test_pred.mean())
print('fraud count:', int(test_pred.sum()))
print(submission.head())

best_t: 0.5900000000000001 / best_oof_f1: 0.897841726618705 / oof_recall: 0.8666666666666667
saved: xgb_tuned_bundle.joblib
fraud rate: 0.0010796766236844185
fraud count: 123
       id  Class
0  170883      0
1  170884      0
2  170885      0
3  170886      0
4  170887      0


In [43]:
for t in [0.20, 0.18, 0.15]:
    pred = (proba >= t).astype(int)

    sub = pd.DataFrame({'id': test['id'], 'Class': pred})
    sub.to_csv(f'sub_{t}.csv', index=False)

    print(t, pred.sum(), pred.mean())

0.2 116 0.0010182316125804272
0.18 116 0.0010182316125804272
0.15 120 0.0010533430474969935
